# Wave 3 — UpstageParser / SiteAParser / SiteBParser

**먼저 `00_setup.ipynb`를 실행하세요.**

## 1. UpstageParser 테스트

In [ ]:
%%ipytest
import os, pytest
from unittest.mock import patch, MagicMock
from langchain_core.documents import Document

UPSTAGE_RESPONSE = {
    'elements': [
        {'content': {'markdown': '## 제목\n내용'}, 'type': 'heading', 'page': 1, 'confidence': 0.99},
        {'content': {'markdown': '단락 텍스트'}, 'type': 'paragraph', 'page': 1, 'confidence': 0.95},
        {'content': {'markdown': ''}, 'type': 'figure', 'page': 2, 'confidence': 0.8},  # 빈 content → 스킵
    ]
}

def make_parser():
    with patch.dict(os.environ, {'UPSTAGE_API_KEY': 'test-key'}):
        from pipeline.parsers.upstage_parser import UpstageParser
        return UpstageParser()

def test_init_raises_without_api_key(tmp_path):
    env = {k: v for k, v in os.environ.items() if k != 'UPSTAGE_API_KEY'}
    with patch.dict(os.environ, env, clear=True):
        from pipeline.parsers.upstage_parser import UpstageParser
        with pytest.raises(EnvironmentError):
            UpstageParser()

def test_maps_elements_to_documents(tmp_path):
    pdf = tmp_path / 'scan.pdf'
    pdf.write_bytes(b'%PDF fake')
    mock_resp = MagicMock()
    mock_resp.json.return_value = UPSTAGE_RESPONSE
    mock_resp.raise_for_status.return_value = None
    parser = make_parser()
    with patch('pipeline.parsers.upstage_parser.requests.post', return_value=mock_resp):
        result = parser.parse(str(pdf))
    assert len(result) == 2  # 빈 element 제외
    assert all(isinstance(d, Document) for d in result)

def test_skips_empty_content(tmp_path):
    pdf = tmp_path / 'scan.pdf'
    pdf.write_bytes(b'%PDF fake')
    mock_resp = MagicMock()
    mock_resp.json.return_value = {'elements': [{'content': {'markdown': ''}, 'type': 'figure', 'page': 1, 'confidence': 0.8}]}
    mock_resp.raise_for_status.return_value = None
    parser = make_parser()
    with patch('pipeline.parsers.upstage_parser.requests.post', return_value=mock_resp):
        result = parser.parse(str(pdf))
    assert result == []

def test_preserves_metadata(tmp_path):
    pdf = tmp_path / 'scan.pdf'
    pdf.write_bytes(b'%PDF fake')
    mock_resp = MagicMock()
    mock_resp.json.return_value = UPSTAGE_RESPONSE
    mock_resp.raise_for_status.return_value = None
    parser = make_parser()
    with patch('pipeline.parsers.upstage_parser.requests.post', return_value=mock_resp):
        result = parser.parse(str(pdf))
    assert result[0].metadata['element_type'] == 'heading'
    assert result[0].metadata['page'] == 1
    assert result[0].metadata['confidence'] == 0.99

## 2. SiteAParser 테스트

In [ ]:
%%ipytest
import os, pytest
from unittest.mock import patch, MagicMock
from langchain_core.documents import Document

with patch.dict(os.environ, {'UPSTAGE_API_KEY': 'test-key'}):
    with patch('pipeline.adapters.site_a.DoclingParser'), patch('pipeline.adapters.site_a.UpstageParser'):
        from pipeline.adapters.site_a import SiteAParser

def test_extract_drawing_id_found():
    assert SiteAParser._extract_drawing_id('도면 DWG-1234 참고') == 'DWG-1234'

def test_extract_drawing_id_not_found():
    assert SiteAParser._extract_drawing_id('번호 없음') == ''

def test_extract_drawing_id_takes_first():
    assert SiteAParser._extract_drawing_id('DWG-1111 DWG-2222') == 'DWG-1111'

def test_parse_manual_adds_site_a():
    with patch.dict(os.environ, {'UPSTAGE_API_KEY': 'test-key'}):
        with patch('pipeline.adapters.site_a.DoclingParser') as MD, \
             patch('pipeline.adapters.site_a.UpstageParser'):
            MD.return_value.parse.return_value = [Document(page_content='내용', metadata={'source': 't.pdf'})]
            parser = SiteAParser()
            result = parser.parse_manual('/data/manual.pdf')
    assert all(d.metadata['site'] == 'A' for d in result)

def test_parse_drawing_skips_empty_pages():
    with patch.dict(os.environ, {'UPSTAGE_API_KEY': 'test-key'}):
        with patch('pipeline.adapters.site_a.DoclingParser'), \
             patch('pipeline.adapters.site_a.UpstageParser'), \
             patch('pipeline.adapters.site_a.fitz.open') as mock_fitz:
            page = MagicMock()
            page.get_text.return_value = ''  # 빈 페이지
            mock_pdf = MagicMock()
            mock_pdf.__iter__ = MagicMock(return_value=iter([page]))
            mock_pdf.close = MagicMock()
            mock_fitz.return_value = mock_pdf
            parser = SiteAParser()
            result = parser.parse_drawing('/data/drawing.pdf')
    assert result == []

## 3. SiteBParser 테스트 (Excel)

In [ ]:
%%ipytest
import os, pytest, openpyxl
from unittest.mock import patch, MagicMock
from langchain_core.documents import Document

def make_xlsx(tmp_path, sheet_data):
    wb = openpyxl.Workbook()
    wb.remove(wb.active)
    for sheet_name, rows in sheet_data.items():
        ws = wb.create_sheet(title=sheet_name)
        for row in rows:
            ws.append(row)
    path = tmp_path / 'test.xlsx'
    wb.save(str(path))
    return str(path)

def make_site_b_parser():
    with patch.dict(os.environ, {'UPSTAGE_API_KEY': 'test-key'}):
        with patch('pipeline.adapters.site_b.DoclingParser'), patch('pipeline.adapters.site_b.UpstageParser'):
            from pipeline.adapters.site_b import SiteBParser
            return SiteBParser()

def test_xlsx_dispatches_to_parse_excel(tmp_path):
    xlsx = make_xlsx(tmp_path, {'Sheet1': [['이름', '값'], ['볼트', 'M10']]})
    parser = make_site_b_parser()
    result = parser.parse_manual(xlsx)
    assert len(result) == 1

def test_parse_excel_formats_as_key_value(tmp_path):
    xlsx = make_xlsx(tmp_path, {'Sheet1': [['이름', '값'], ['볼트', 'M10']]})
    parser = make_site_b_parser()
    result = parser._parse_excel(xlsx)
    assert '이름: 볼트' in result[0].page_content
    assert '값: M10' in result[0].page_content

def test_parse_excel_skips_empty_rows(tmp_path):
    xlsx = make_xlsx(tmp_path, {'Sheet1': [['이름', '값'], [None, None], ['볼트', 'M10']]})
    parser = make_site_b_parser()
    result = parser._parse_excel(xlsx)
    assert len(result) == 1

def test_parse_excel_multiple_sheets(tmp_path):
    xlsx = make_xlsx(tmp_path, {'Sheet1': [['항목'], ['값A']], 'Sheet2': [['항목'], ['값B']]})
    parser = make_site_b_parser()
    result = parser._parse_excel(xlsx)
    assert len(result) == 2

def test_parse_excel_sets_doc_type_spec(tmp_path):
    xlsx = make_xlsx(tmp_path, {'Sheet1': [['항목', '값'], ['온도', '200℃']]})
    parser = make_site_b_parser()
    result = parser._parse_excel(xlsx)
    assert all(d.metadata['doc_type'] == 'spec' for d in result)